
# From Images to Text: Working with OCR

## Overview

This course provides an overview of Optical Character Recognition (OCR), an image processing technique for extracting text from images.

We will cover the followings:

#### A. Before OCR: inspect the image and preprocessing
- Inspect the data
- Different types of OCR noise
- Use OCR library to clean up the data, such as removing stains, croping, etc. 

#### B. Applying OCR: Choose the right methods
- language packages
- vertical vs. horezontal text
- text blocks and colunms

#### C. After OCR: clean up the text
- Clean up the missing data and erros using Regex (optional)

> 👋 Let's get you started by running the code block below to make sure everything is working properly for you.

In [ ]:
print("Hello, welcome to the Image to Text with OCR workshop!")



---



# What is OCR?

**O**ptical **C**haracter **R**ecognition
- OCR is a technique to process images of text, such as written or printed documents, and produce **machine-readable** documents.
- Machine-readable documents are encoded in formats that computers can process, which allow the text to be searched, edited and analysed computationally

**Workflow**

1.   **Document selection**: An image or text document needs to be selected to scan.
2.   **Scanning**: The images are then scanned by OCR software. This generates a machine-readable text output.
3.   **Cleaning** :Once you have an OCR output, extra steps to clean the files might be need to improve accuracy.
4.   **Saving files**: the machine-readable documents can be saved and are to use.



### **Challange and Error of OCR**

You are likely to encounter problems or errors in the text outputs:

The accuracy of OCR can be limited by the **OCR engine capability**, such as:
- file size
- format of the input files
- text orientation
- language

It also depends on **dataset quality, formatting of the origianl documents:**
- Human errors and typos​
- Age and damage (stained or blurry)​
- Mixed text and images, or multiple languages​
- Cursive handwriting​

## ✏️ *Activity: Online OCR Engine*  









 Choose a pdf that contains text and try uploading it into some of these online programmes:
*   https://tools.pdf24.org/en/ocr-pdf   
*   https://www.onlineocr.net/   
*   https://www.sodapdf.com/ocr-pdf/   
*   https://www.sejda.com/ocr-pdf   
*   https://ocr.space/   
*   https://avepdf.com/pdf-ocr   

## Ways to improve OCR accuracy

- Select good quality dataset to begin with​

- Improve OCR tool capability​

- Pre-process your dataset to improve its quality​

- Correct errors in OCR-produced files

---

# 1. Inspect the images & Pre-processing

#### Factors affecting OCR results

- **Image colour:** colour images can be used, but ensure there is a sharp contrast between the background page and the text itself (for example black text on white or cream paper).

- **Text orientation:** some languages can be written vertically, such as Japanses, and Chinese; right to left, such as Arabic. The original data could be distorted or tilted, these could affect the accuracy of OCR. 

- **Languages:** Most OCR engines or packages require users to choose the language of the data. Some can detect multiple langauges at once, but they might not be as accurate as using single language.

- **File format:**
Some OCR engines will only accept certain file formats, so it is best to check this in advance; some will only use image files, whereas others will work on PDF files. 
**TIFF** files are lossless files, meaning that no image quality or information is lost; this means that TIFF images are usually high quality but also much larger file sizes. PNGs are also lossless, although TIFF files would be preferred over PNGs for OCR.
**JPEGs** are a lossy format type, meaning that the image is compressed to create a smaller file size. OCR engines can work with JPEGs, but there may be a loss of image quality that can impact the text generation.

## ✏️ Activity: Immarkus








IMMARKUS is a free, open-source, browser-based tool designed to help researchers and students analyse, annotate and extract information from images. It is particularly useful for digital humanities, historical documents, archival materials and cultural heritage collections. 

Website: [IMMARKUS](https://immarkus.xmarkus.org/)

Sample dataset: [Kaggle OCR scanned images](https://www.kaggle.com/datasets/suvroo/scanned-images-dataset-for-ocr-and-vlm-finetuning?resource=download)

# 🟢 **OCR with Tesseract (pytesseract)**

Tesseract is an OCR engine developed for various operating systems​ to extract printed text from images. It is of the most accurate open-source OCR engines available​.


## ✏️ *Activity: pytesseract*  

In [ ]:
### Initialisation and setting up the environment will take abuot one minutes, 

!pip install pytesseract
#!conda install -c conda-forge tesseract pytesseract -y

import pandas as pd
import os

# These are the packages you will need to process the image and perform OCR scanning
import pytesseract # OCR engine tesseract
import cv2 # opencv
import numpy as np # supporting open cv

# To load language packages
import os
os.environ['TESSDATA_PREFIX'] = 'data/tessdata/'
#print(pytesseract.get_languages(config='')) show available language packages

# To plot and show image
from skimage import io
from PIL import Image, ImageDraw
from IPython.display import display

# To compare the OCR text with the reference text and calculate similarity
from difflib import SequenceMatcher
from pathlib import Path
import unicodedata

def compare_ocr_text(
    text: str,
    path: str = "data/results/text_result1.txt"
) -> tuple[str, float]:
    """Return text with word differences marked, plus similarity percentage."""
    reference_words = Path(path).read_text(encoding="utf-8").split()
    clean = lambda word: "".join(c for c in word if not unicodedata.category(c).startswith("P"))
    text_words = text.split()
    matcher = SequenceMatcher(None, list(map(clean, text_words)), list(map(clean, reference_words)))
    percentage = matcher.ratio() * 100 if text_words or reference_words else 100.0
    annotated_words = []

    for tag, text_start, text_end, ref_start, ref_end in matcher.get_opcodes():
        wrong, correct = text_words[text_start:text_end], reference_words[ref_start:ref_end]
        if tag == "equal":
            annotated_words.extend(wrong)
        elif tag == "replace":
            n = min(len(wrong), len(correct))
            annotated_words.extend(f"{a}[{b}]" for a, b in zip(wrong[:n], correct[:n]))
            annotated_words.extend(f"{w}[]" for w in wrong[n:])
            annotated_words.extend(f"[{c}]" for c in correct[n:])
        elif tag == "delete":
            annotated_words.extend(f"{w}[]" for w in wrong)
        elif tag == "insert":
            annotated_words.extend(f"[{c}]" for c in correct)

    return " ".join(annotated_words), percentage

## Basic text recognision with Tesseract

We use Pillow `Image.open()` to open and display an image. Let's see what is in the picture:

In [ ]:
img_english = Image.open("data/img_english.jpg") # Load and display the image
display(img_english)

Next, we will extract the text from the image using pytesseract library.
The function `pytesseract.image_to_string` takes an image as input and returns the recognized text.

In [ ]:
# Perform OCR on the image, and store the extracted text in ocr_text
ocr_text = pytesseract.image_to_string(img_english) 

print("\nHere is the result:\n")
print(ocr_text)

## Text recognision in different/multiple languages



Pytesseract uses English as its default language recognition package. However, more and more packages have been developed and you can find information and download the packages from the following website:

- Check language package information: [Here](https://tesseract-ocr.github.io/tessdoc/Data-Files-in-different-versions.html)
- Download language packages: [Here](https://github.com/tesseract-ocr)
- The most commonly used pre-trained language packages can be found in  tessdata_best folder


Once you've downloaded the packages, you can check if they are installed successfully.
The following block shows all available tesseract language packages represented by their package codes

In [ ]:
print(pytesseract.get_languages(config=''))

Load and show image with Traditional Chinese.
Or Spanish text (by removing # at the top of each line)

In [ ]:
img_traditional_chinese = Image.open("data/img_chi_tra.jpg")
display(img_traditional_chinese)

#img_spanish = Image.open("data/img_spa.jpg")
#display(img_spanish)

#### 🔥 EXERCISE

> Extract text by specifying the language package uisng argument `lang=''`

> Please replace `???` with the right language package

In [ ]:
# Hint: 
# 1. Choose the image for OCR (img_chi_tra.jpg, or img_spa.jpg)
# 2. inter the right language package in lang = '', for example, 'chi_tra', or 'spa'

img = Image.open("data/???.jpg")
eng_ocr_text = pytesseract.image_to_string(img, lang='eng') 
ocr_text = pytesseract.image_to_string(img, lang='???') 

print("\nHere is the result:\n")
print("Without specifying language:")
print(eng_ocr_text)
print("\nWith language specified:")
print(ocr_text)

In [ ]:
#load and show image with multiple languages
img_multi = Image.open("data/img_multi.jpg")
display(img_multi)

#### 🔥 EXERCISE

> Specify the languages that Tesseract should recognize.

In [ ]:
# Hint: For multiple languages, join the language codes with a `+`` sign.
#       In our case, the language would be 
#       Spanish 'spa', Japanese 'jpan', and English 'eng'

img_multi = Image.open("data/img_multi.jpg")
ocr_text = pytesseract.image_to_string(img_multi, lang='???') #the extracted text is stored in ocr_text
print("\nHere is the result:\n")
print(ocr_text)

Note that the the time taken for OCR as well as the output can be different based on the order of languages. That means, using "jpan + eng + spa" might get a different result from "spa + jpan +eng".
It is recommended to start with the main language in the images.

#### 🔥 EXERCISE: Try extracting text from multiple colunms

In [ ]:
img_blocks = Image.open("data/img_blocks.jpg")
display(img_blocks)

In [ ]:
ocr_text = pytesseract.image_to_string(img_blocks,lang="eng")

print("\nHere is the result:\n")
print(ocr_text)


## Extra: Specifying segmentation

By default Tesseract expects a page of text when it segments an image. If you’re just seeking to OCR a small region, try a different segmentation mode, using the --psm argument.

```
  0    Orientation and script detection (OSD) only.
  1    Automatic page segmentation with OSD.
  2    Automatic page segmentation, but no OSD, or OCR.
  3    Fully automatic page segmentation, but no OSD. (Default)
  4    Assume a single column of text of variable sizes.
  5    Assume a single uniform block of vertically aligned text.
  6    Assume a single uniform block of text.
  7    Treat the image as a single text line.
  8    Treat the image as a single word.
  9    Treat the image as a single word in a circle.
 10    Treat the image as a single character.
 11    Sparse text. Find as much text as possible in no particular order.
 12    Sparse text with OSD.
 13    Raw line. Treat the image as a single text line,
```
- see full documentation [Here](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html#page-segmentation-method)
- useful explanation of all options [here](https://pyimagesearch.com/2021/11/15/tesseract-page-segmentation-modes-psms-explained-how-to-improve-your-ocr-accuracy/)(some of them might not work properly)


## Colour correction


For OCR, clearer images can make it easier for the OCR engines to identify letters and words accurately. 
Colour correction can help by increasing contrast, and making faded text more visible. It can also make images more consistent when processing a collection of documents.

Let's have a look at the following image that is partly covered with shadows:

In [ ]:
img_shadow = Image.open('data/img_shadow.jpg')
display(img_shadow)
ocr_text = pytesseract.image_to_string(img_shadow,lang="eng")

print("\nHere is the result:\n")
print(ocr_text)


Run the following block to see the word-level OCR accuracy

In [ ]:
annotated_text, percentage = compare_ocr_text(ocr_text)
print(annotated_text)
print(f"Word-level similarity: {percentage:.2f}%")

The results above shows some of the text is not identified correctly due to the shadows. 

This can be fixed by applying colour correction methods using Open cv before OCR.
Here we are going to use open cv as an example to pre-process the images and make them become clearer. 
Below are commonly used steps to make images clearer: 

Start with colour convertion: 
- COLOR_BGR2GRAY conver image to grey scale
- Open cv [Colour conversion code](https://docs.opencv.org/3.4/d8/d01/group__imgproc__color__conversions.html)

Use thresholding to improve contrast:
- Open cv [Threshholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)


#### 🔥 EXERCISE

> Run the folloiwng block and compare image before and after colour correction.

> After loading the image, we then converts the image to a NumPy array, using `np.array`, so it now holds the array rather than the original PIL image. The `cv2.COLOR_BGR2GRAY` turns the image to gray scale

In [ ]:
img_shadow = Image.open('data/img_shadow.jpg')

gray_img = cv2.cvtColor(np.array(img_shadow), cv2.COLOR_BGR2GRAY)
display(Image.fromarray(gray_img)) 


### Thresholding method

`cv2.adaptiveThreshold(gray_img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, ? ,?)`

- The 1st argument is the source image, which should be a  grayscale image.
- The 2nd argument is the threshold value which is used to classify the pixel values.
- The 3rd argument is the adaptive thresholding method (here we use mean value method)
- The 4th arguement is the threshold method (here we use binary method, same to simple thresholding)
- The 5th our pixel neighborhood size, it must be an odd number
- The 6th arguement let you tune the thresholding result. (more infromation on how to choose this constant [here](https://pyimagesearch.com/2021/05/12/adaptive-thresholding-with-opencv-cv2-adaptivethreshold/))

In [ ]:
from os import name

img_shadow = Image.open('data/img_shadow.jpg')
gray_img = cv2.cvtColor(np.array(img_shadow), cv2.COLOR_BGR2GRAY)
gray_img_big = cv2.resize(gray_img,None,fx=2,fy=2,interpolation=cv2.INTER_CUBIC)
ocr_text = pytesseract.image_to_string(gray_img_big,lang="eng")
annotated_text, percentage = compare_ocr_text(ocr_text)
print(annotated_text)
print(f"Without thresholding - Word-level similarity: {percentage:.2f}%")


best = None


for block in [7,21]:
    for c in range(22,26):
        thresh = cv2.adaptiveThreshold(
            gray_img_big, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block, c
        )
        text = pytesseract.image_to_string(thresh)
        annotated, score = compare_ocr_text(text)

        if best is None or score > best[0]:
            best = (score, name, block, c, thresh, annotated)

            
score, method, block, c, thresh, annotated = best
print(f"After threholdeing - Best: {method}, blockSize={block}, C={c} — {score:.2f}%")
print(annotated)
display(Image.fromarray(thresh))


---

# Example 1: verticle text


### 01 - Colour correction with thresholding technique
Thresholding is the binarization of an image. We want to:
1. convert a coloured image to grayscale image


2. convert the grayscale image to a binary image, where the pixels are either 0 or 255

This can be easily done with image processing library such as Opencv



In [ ]:
# load and show image
img_japn_vert = Image.open("data/img_japn_vert.jpg")
display(img_japn_vert)

In [ ]:
# Convert image to grayscale using the same method mentioned above
img_japn_vert = Image.open("data/img_japn_vert.jpg")
img_shadow = cv2.cvtColor(np.array(img_japn_vert), cv2.COLOR_RGB2BGR)
gray_img = cv2.cvtColor(img_shadow, cv2.COLOR_BGR2GRAY)

In [ ]:
# Perform simple thresholding, 
# There’s no universally best pair;
# it depends on the image’s lighting, text size, and background.
# For cv2.THRESH_BINARY, pixels above the calculated threshold become white; the others become black.

thresh = cv2.adaptiveThreshold(
    gray_img,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    ?,  # must be an odd number greater than 1
    ?   # Adjusting this parameter changes how much contrast is needed for 
        # a pixel to become white or black.
)

display(Image.fromarray(thresh))

In [ ]:
# Reference result
img = Image.open('data/results/thresh_result2.jpg')
display(img)


### 02 - Recognise verticle text
For languages that use verticle writing system, there are language packages trained to tackle these variations.

You can visit Tesseract documentation [page](https://tesseract-ocr.github.io/tessdoc/Data-Files-in-different-versions.html) to check language package information and find the language package for recognising vertical japanese text.

Also, you can play with different segmentation option and compare the results:


In [ ]:
ocr_text = pytesseract.image_to_string(thresh, lang='jpan_vert', config="--psm 5")
print("\nHere is the result:\n")
print(ocr_text)


Reference results

```
可愛 い 1 年 生

小学 校 で 教員 を し て いる 。
体育 の 授業 で 転ん で し まっ た
1 年 生 に ば ん そう こう を 則っ
て あげ た 。 放 課 後 学童 保育
ど に 寄る か 確認 し た く て [ 今
日 まっ すぐ 帰る ?」 と 聞く <
と 、 泣 き な が ら 「 い な V え 、 ち
ょ っ と 曲がり ます 」。
(川崎 市 ・ 曲 が ら な きゃ 帰れ
な いも の ね ・ 公 歳 )
```



# Example 2: Meme with different colour background

> Please follow the following workflow to complete the task


### 01 - Colour selection with masking technique
This example demonstrates a different colour convertion technique by creating a colour filter to make the range of colour of the text we want to extract from the image.



In [ ]:
img_meme = Image.open('data/img3.jpg')
display(img_meme)

In [ ]:
text = pytesseract.image_to_string(img_meme, lang='eng')
print(text)

The result shows that only the text "Yellow Text with Black Background" was extracted. Let's try increase the contrast of the colours with opencv.

In [ ]:
img_array = np.array(img_meme)
HSV = cv2.cvtColor(img_array, cv2.COLOR_BGR2HSV)
display(Image.fromarray(HSV))

In [ ]:
# Adjust lower and upper while viewing the mask until it selects the color you want. 
# The example are just a starting point for green
# the right range depends on the color in your image.

# H (Hue): the color itself, such as red, green, or blue.
# S (Saturation): how vivid or muted the color is.
# V (Value): how bright or dark it is.

lower = np.array([35, 40, 80])
upper = np.array([90, 255, 255])
mask = cv2.inRange(HSV, lower, upper) #Define the colour range
display(Image.fromarray(mask))



In [ ]:
imask = mask > 0 #Select the pixels where the colour is in the range
result = np.zeros_like(img_array, np.uint8) #Create an empty image the same size to the file
result[imask] = img_array[imask] #Copy the part of image in the colour range to the resulting image
display(Image.fromarray(result))
data = pytesseract.image_to_string(result, lang='eng')
print(data)

## More application of Open CV and tesseract with line detection for forms

- [Edge and line detection](https://pyimagesearch.com/2015/04/06/zero-parameter-automatic-canny-edge-detection-with-python-and-opencv/)
- [How to detect table](https://livefiredev.com/how-to-extract-table-from-image-in-python-opencv-ocr/)



---



# **Resources List**

**Tutorials**

Centre for Data, Culture and Society. "Text Extraction & Preparation," Managing Digitised Documents (2022), https://www.cdcs.ed.ac.uk/training/training-pathways/managing-digitised-documents-pathway [accessed 23 July 2023].

Knox, Doug. "Understanding Regular Expressions," Programming Historian 2 (2013), https://doi.org/10.46430/phen0033 [accessed 23 July 2023].

Turner O'Hara, Laura. "Cleaning OCR’d text with Regular Expressions," Programming Historian 2 (2013), https://doi.org/10.46430/phen0024 [accessed 23 July 2023].

Library Carpentry. "Introduction to Working with Data (Regular Expressions)" (2023), https://librarycarpentry.org/lc-data-intro/01-regular-expressions.html [accessed 23 July 2023].

"Regular Expressions Cheat Sheet" by Davidchild, https://cheatography.com/davechild/cheat-sheets/regular-expressions/ [accessed 10 Sep 2025]

**Readings**

Cordell, Ryan. ‘“Q i-Jtb the Raven”: Taking Dirty OCR Seriously’. Book History 20, no. 1 (2017): 188–225. https://doi.org/10.1353/bh.2017.0006 [accessed 23 July 2023].

Schantz, Herbert F. The History of OCR, Optical Character Recognition. [Manchester Center, Vt.] : Recognition Technologies Users Association, (1982) http://archive.org/details/historyofocropti0000scha [accessed 23 July 2023].
